In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from core.utils.device import DEVICE
from core.utils.theme import set_theme

set_theme()

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
model.to(DEVICE)
model.eval()
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression

from core.steering.pca import PCASteering


def projections_to_Xy(projections, langs, num_of_pca_components):
    lang1_data = projections[langs[0]][-1].numpy()
    lang2_data = projections[langs[1]][-1].numpy()
    X = np.vstack([lang1_data, lang2_data])
    y = np.array([1] * lang1_data.shape[0] + [0] * lang2_data.shape[0])
    X = X[:, :num_of_pca_components]

    return X, y


def train_and_score(projections_train, projections_test, num_of_pca_components=1):
    """
    hidden_space_by_language: { [lang]: torch.Tensor([n_layers, n_tokens, d_model]) }
    """
    langs = list(projections_train.keys())
    X_trn, y_trn = projections_to_Xy(projections_train, langs, num_of_pca_components)
    X_tst, y_tst = projections_to_Xy(projections_test, langs, num_of_pca_components)

    classifier = LogisticRegression(random_state=42, penalty="l2")

    classifier.fit(X_trn, y_trn)
    train_accuracy = classifier.score(X_trn, y_trn)
    test_accuracy = classifier.score(X_tst, y_tst)

    print(
        f"# of PCA components: {num_of_pca_components}, train accuracy {train_accuracy:.4f}, test accuracy {test_accuracy:.4f}"
    )

# EN-RU


In [4]:
from core.gather_data.hidden_space import collect_hidden_space_by_language
from core.preprocess_data.flores_plus import load_flores_plus

train_df, test_df = load_flores_plus(["eng_Latn", "rus_Cyrl"], {"eng_Latn": "en", "rus_Cyrl": "ru"}, train_size=50)

test_df = test_df[:100]

hidden_space_by_language_train, token_map_for_language_train = collect_hidden_space_by_language(
    model, tokenizer, train_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_ru_flores_cls_train.pt"
)
hidden_space_by_language_test, token_map_for_language_test = collect_hidden_space_by_language(
    model, tokenizer, test_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_ru_flores_cls_test.pt"
)

pca_steering = PCASteering(cache_path="../../../.cache/pca/llama_en_ru_flores_cls.pt").fit(hidden_space_by_language_train)

projections_train = pca_steering.project(hidden_space_by_language_train)
projections_test = pca_steering.project(hidden_space_by_language_test)

train_and_score(projections_train, projections_test)

Data len:  50


100%|██████████| 50/50 [00:23<00:00,  2.15it/s]


Data len:  100


100%|██████████| 100/100 [01:23<00:00,  1.20it/s]
/src/language-steering-in-latent-space/src/core/steering/pca.py:67: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.pca_components = torch.tensor(pca_components)


# of PCA components: 1, train accuracy 0.9924, test accuracy 0.9898


# EN-CN


In [5]:
from core.gather_data.hidden_space import collect_hidden_space_by_language
from core.preprocess_data.flores_plus import load_flores_plus
from core.steering.pca import PCASteering

train_df, test_df = load_flores_plus(["eng_Latn", "cmn_Hans"], {"eng_Latn": "en", "cmn_Hans": "cn"}, train_size=50)

test_df = test_df[:100]

hidden_space_by_language_train, token_map_for_language_train = collect_hidden_space_by_language(
    model, tokenizer, train_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_cn_flores_cls_train.pt"
)
hidden_space_by_language_test, token_map_for_language_test = collect_hidden_space_by_language(
    model, tokenizer, test_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_cn_flores_cls_test.pt"
)

pca_steering = PCASteering(cache_path="../../../.cache/pca/llama_en_cn_flores_cls.pt").fit(hidden_space_by_language_train)

projections_train = pca_steering.project(hidden_space_by_language_train)
projections_test = pca_steering.project(hidden_space_by_language_test)

train_and_score(projections_train, projections_test)

Data len:  50


100%|██████████| 50/50 [00:29<00:00,  1.69it/s]


Data len:  100


100%|██████████| 100/100 [01:39<00:00,  1.00it/s]


# of PCA components: 1, train accuracy 0.9917, test accuracy 0.9842


# EN-ES


In [6]:
from core.gather_data.hidden_space import collect_hidden_space_by_language
from core.preprocess_data.flores_plus import load_flores_plus
from core.steering.pca import PCASteering

train_df, test_df = load_flores_plus(["eng_Latn", "spa_Latn"], {"eng_Latn": "en", "spa_Latn": "es"}, train_size=50)

test_df = test_df[:100]

hidden_space_by_language_train, token_map_for_language_train = collect_hidden_space_by_language(
    model, tokenizer, train_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_es_flores_cls_train.pt"
)
hidden_space_by_language_test, token_map_for_language_test = collect_hidden_space_by_language(
    model, tokenizer, test_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_es_flores_cls_test.pt"
)

pca_steering = PCASteering(cache_path="../../../.cache/pca/llama_en_es_flores_cls.pt").fit(hidden_space_by_language_train)

projections_train = pca_steering.project(hidden_space_by_language_train)
projections_test = pca_steering.project(hidden_space_by_language_test)

train_and_score(projections_train, projections_test)

Data len:  50


100%|██████████| 50/50 [00:33<00:00,  1.51it/s]


Data len:  100


100%|██████████| 100/100 [01:37<00:00,  1.03it/s]


# of PCA components: 1, train accuracy 0.9648, test accuracy 0.9615


# EN-HIN

In [7]:
from core.gather_data.hidden_space import collect_hidden_space_by_language
from core.preprocess_data.flores_plus import load_flores_plus
from core.steering.pca import PCASteering

train_df, test_df = load_flores_plus(["eng_Latn", "hin_Deva"], {"eng_Latn": "en", "hin_Deva": "hin"}, train_size=50)

test_df = test_df[:100]

hidden_space_by_language_train, token_map_for_language_train = collect_hidden_space_by_language(
    model, tokenizer, train_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_hin_flores_cls_train.pt"
)
hidden_space_by_language_test, token_map_for_language_test = collect_hidden_space_by_language(
    model, tokenizer, test_df, skip_first=True, cache_path="../../../.cache/hidden_space/llama_en_hin_flores_cls_test.pt"
)

pca_steering = PCASteering(cache_path="../../../.cache/pca/llama_en_hin_flores_cls.pt").fit(hidden_space_by_language_train)

projections_train = pca_steering.project(hidden_space_by_language_train)
projections_test = pca_steering.project(hidden_space_by_language_test)

train_and_score(projections_train, projections_test)

Data len:  50


100%|██████████| 50/50 [00:27<00:00,  1.81it/s]


Data len:  100


100%|██████████| 100/100 [01:45<00:00,  1.05s/it]


# of PCA components: 1, train accuracy 0.9969, test accuracy 0.9957
